In [1]:
import os
import glob
import re
from nbconvert import HTMLExporter
from traitlets.config import Config
import nbformat

# Install beautifulsoup4 if not already installed
try:
    from bs4 import BeautifulSoup
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'beautifulsoup4'])
    from bs4 import BeautifulSoup

def update_ipynb_links_to_html(html_body):
    """
    Post-process HTML body to:
    1. Replace .ipynb links with .html equivalents, preserving fragments (#anchors).
    2. Replace YouTube thumbnail images with embedded iframes.
    
    Args:
        html_body (str): The HTML content as a string.
    
    Returns:
        str: Updated HTML body.
    """
    # First, handle .ipynb link replacement (original functionality)
    pattern = r'(href=")([^"]*\.)?(?P<before>[^"#\.]*)ipynb(?P<after>#?[^"]*)"'
    def replacer(match):
        # Replace .ipynb with .html, preserving path, before, and after (including #anchor)
        return f'{match.group(1)}{match.group(2)}{match.group("before")}html{match.group("after")}"'
    
    updated_body = re.sub(pattern, replacer, html_body)
    
    # Second, handle YouTube thumbnail replacement with iframes
    soup = BeautifulSoup(updated_body, 'html.parser')
    replacements = 0
    
    # Find all <a> tags with YouTube video links containing <img> thumbnails
    for a_tag in soup.find_all('a', href=re.compile(r'youtube\.com/watch\?v=([a-zA-Z0-9_-]+)')):
        img_tag = a_tag.find('img')
        if img_tag and 'src' in img_tag.attrs:
            img_src = img_tag['src']
            # Extract VIDEO_ID from href (watch?v=VIDEO_ID)
            video_match = re.search(r'youtube\.com/watch\?v=([a-zA-Z0-9_-]+)', a_tag['href'])
            if video_match:
                video_id = video_match.group(1)
                # Check if src is YouTube thumbnail (vi/VIDEO_ID/hqdefault.jpg or similar)
                if re.search(rf'img\.youtube\.com/vi/{re.escape(video_id)}/[a-z]+default\.jpg', img_src):
                    # Create responsive iframe wrapper
                    responsive_div = soup.new_tag('div', style="position: relative; padding-bottom: 56.25%; height: 0; overflow: hidden; max-width: 100%; margin: 20px 0;")
                    responsive_div['class'] = 'youtube-embed'
                    
                    # Create iframe with your specified attributes
                    iframe = soup.new_tag('iframe',
                                          width="560",
                                          height="315",
                                          src=f"https://www.youtube.com/embed/{video_id}",
                                          title="YouTube video player",
                                          frameborder="0",
                                          allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share",
                                          referrerpolicy="strict-origin-when-cross-origin",
                                          allowfullscreen=True)
                    
                    # Make iframe responsive within the div
                    iframe['style'] = "position: absolute; top: 0; left: 0; width: 100%; height: 100%;"
                    responsive_div.append(iframe)
                    
                    # Replace the entire <a><img></a> with the responsive iframe div
                    a_tag.replace_with(responsive_div)
                    replacements += 1
                    print(f"Replaced YouTube thumbnail ({video_id}) with embedded iframe")
    
    if replacements > 0:
        print(f"Total YouTube thumbnails replaced with iframes: {replacements}")
    
    return str(soup)

def convert_ipynb_to_html_with_link_updates(directory_path):
    """
    Recursively convert all .ipynb files to HTML, preserving relative images and updating .ipynb links (with anchors) to .html.
    Also replaces YouTube thumbnail images with embedded iframes.
    Special handling: Rename "OPEN ME FIRST - README - Main Menu.ipynb" output to root index.html.
    
    Args:
        directory_path (str): Path to the root directory to search for .ipynb files.
    """
    # Find all .ipynb files recursively (sorted for consistent order)
    ipynb_files = sorted(glob.glob(os.path.join(directory_path, '**', '*.ipynb'), recursive=True))
    
    if not ipynb_files:
        print("No .ipynb files found in the directory.")
        return
    
    # Configure exporter for relative paths (no embedding)
    c = Config()
    c.HTMLExporter.embed_images = False
    exporter = HTMLExporter(config=c)
    
    root_dir = os.path.abspath(directory_path)
    target_notebook_name = "OPEN ME FIRST - README - Main Menu.ipynb"  # Exact filename with spaces
    
    for ipynb_path in ipynb_files:
        try:
            # Read the notebook
            with open(ipynb_path, 'r', encoding='utf-8') as f:
                notebook = nbformat.read(f, as_version=4)
            
            # Convert to HTML
            body, resources = exporter.from_notebook_node(notebook)
            
            # Post-process: Update .ipynb links to .html AND replace YouTube thumbnails with iframes
            body = update_ipynb_links_to_html(body)
            
            # Determine output path
            notebook_name = os.path.basename(ipynb_path)
            if notebook_name == target_notebook_name:
                # Special case: Rename to root index.html
                html_filename = 'index.html'
                html_path = os.path.join(root_dir, html_filename)
                resource_dir_name = 'index_files'  # For attachments
                print(f"Special conversion: Renaming to root {html_path}")
            else:
                # Standard: .html in same dir
                html_dir = os.path.dirname(ipynb_path)
                html_filename = os.path.splitext(notebook_name)[0] + '.html'
                html_path = os.path.join(html_dir, html_filename)
                resource_dir_name = f"{os.path.splitext(html_filename)[0]}_files"
            
            # Write updated HTML file
            with open(html_path, 'w', encoding='utf-8') as f:
                f.write(body)
            
            # Extract attachments to _files if present
            if resources and 'files' in resources:
                if notebook_name == target_notebook_name:
                    resource_dir = os.path.join(root_dir, resource_dir_name)
                else:
                    resource_dir = os.path.join(os.path.dirname(html_path), resource_dir_name)
                os.makedirs(resource_dir, exist_ok=True)
                for file_path, content in resources.get('files', {}).items():
                    with open(os.path.join(resource_dir, file_path), 'wb') as res_file:
                        res_file.write(content)
                print(f"Extracted attachments to: {resource_dir}")
            
            print(f"Converted with YouTube embeds: {ipynb_path} -> {html_path}")
            
        except Exception as e:
            print(f"Error converting {ipynb_path}: {str(e)}")

# Example usage: Replace '.' with your directory path
directory = '.'  # Current directory; change as needed
convert_ipynb_to_html_with_link_updates(directory)


Converted with YouTube embeds: .\CAD_Files\3 Port Reservoir\Manufacturing Notes.ipynb -> .\CAD_Files\3 Port Reservoir\Manufacturing Notes.html
Converted with YouTube embeds: .\ChronoSeq_Overview.ipynb -> .\ChronoSeq_Overview.html
Replaced YouTube thumbnail (qbhCONpUdx0) with embedded iframe
Replaced YouTube thumbnail (J2O1kA4chU0) with embedded iframe
Replaced YouTube thumbnail (qFouf2HrKSk) with embedded iframe
Replaced YouTube thumbnail (QpMUR4AgpgA) with embedded iframe
Total YouTube thumbnails replaced with iframes: 4
Special conversion: Renaming to root C:\Users\ChronoSeq\ChronoSeq\index.html
Converted with YouTube embeds: .\OPEN ME FIRST - README - Main Menu.ipynb -> C:\Users\ChronoSeq\ChronoSeq\index.html
Converted with YouTube embeds: .\convertohtml.ipynb -> .\convertohtml.html
Converted with YouTube embeds: .\instructions_for_assembling_sensirion_flow_sensor.ipynb -> .\instructions_for_assembling_sensirion_flow_sensor.html
Converted with YouTube embeds: .\instructions_for_asse

C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted with YouTube embeds: .\instructions_for_assembling_valve_controller_ESP32.ipynb -> .\instructions_for_assembling_valve_controller_ESP32.html
Converted with YouTube embeds: .\instructions_for_assembling_vortex_relay_controller.ipynb -> .\instructions_for_assembling_vortex_relay_controller.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted with YouTube embeds: .\instructions_for_assembling_xy_robot_and_ice_boxes.ipynb -> .\instructions_for_assembling_xy_robot_and_ice_boxes.html
Converted with YouTube embeds: .\instructions_for_assembling_xyz_robot.ipynb -> .\instructions_for_assembling_xyz_robot.html
Converted with YouTube embeds: .\instructions_for_device_assembly_and_setup.ipynb -> .\instructions_for_device_assembly_and_setup.html
Converted with YouTube embeds: .\instructions_for_setting_coordinates_for_xyz_robot.ipynb -> .\instructions_for_setting_coordinates_for_xyz_robot.html
Converted with YouTube embeds: .\instructions_for_setting_up_valves_tubing_and_reservoirs.ipynb -> .\instructions_for_setting_up_valves_tubing_and_reservoirs.html
Converted with YouTube embeds: .\protocol_and_software_for_running_chronoseq_device-OriginalDevice.ipynb -> .\protocol_and_software_for_running_chronoseq_device-OriginalDevice.html
Converted with YouTube embeds: .\protocol_and_software_for_running_chronoseq_device.ipynb -> .

...

Converted with YouTube embeds: .\protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification.ipynb -> .\protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted with YouTube embeds: .\protocol_for_single_cell_scale_up_chronoseqv5_dropseq_bead_modification.ipynb -> .\protocol_for_single_cell_scale_up_chronoseqv5_dropseq_bead_modification.html
Converted with YouTube embeds: .\protocol_for_tagmentation_with_KAPA_PCR.ipynb -> .\protocol_for_tagmentation_with_KAPA_PCR.html
Converted with YouTube embeds: .\protocol_library_preparation_for_dropseq_chronoseq_beads.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads.html
Converted with YouTube embeds: .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.html
Converted with YouTube embeds: .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.ipynb -> .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted with YouTube embeds: .\qPCR_validation_files\qPCR Analysis.ipynb -> .\qPCR_validation_files\qPCR Analysis.html
Converted with YouTube embeds: .\removeLinksFromNBConvertHTML.ipynb -> .\removeLinksFromNBConvertHTML.html


In [2]:
import os

def generate_sitemap_html(directory_path, output_file='sitemap.html', debug=False):
    """
    Generate a professional HTML site map listing all .html files, categorized by filename patterns and directory structure.
    Includes "Main Hardware Pages", "Data Processing and Analysis" (with repo links), and other sections.
    - "instructions*" files under "ChronoSeq Hardware Assembly".
    - "protocol*" files under "ChronoSeq Protocols".
    - Others under "Other Notebooks" (hierarchical by dir, including qPCR_validation_files and _files folders).
    Ignores only ivPID directory and specified basenames: ChronoSeq_Overview, convertToHTML, index, removeLinksFromNBConvertHTML, sitemap.
    
    Args:
        directory_path (str): Path to the root directory to scan for .html files.
        output_file (str): Name of the output HTML file (default: sitemap.html in root).
        debug (bool): If True, print found .html paths (including qPCR_validation_files and _files specifics) to console.
    """
    root_dir = os.path.abspath(directory_path)
    
    # Ignored filenames (case-insensitive, without .html extension) - convertToHTML is explicitly here
    ignored_basenames = {
        'chronoseq_overview', 'convertohtml', 'index', 
        'removelinksfromnbconverthtml', 'sitemap'
    }
    
    # Manual main hardware links (adjust paths if not in root)
    main_hardware_links = {
        'assembly': 'instructions_for_device_assembly_and_setup.html',
        'operation': 'protocol_and_software_for_running_chronoseq_device.html'  # Corrected for operation page
    }
    
    # ChronoSeq repo URL for linking all mentions
    chronoseq_repo = 'https://github.com/kanishkasthana/ChronoSeq'
    
    # Collect all .html files recursively, excluding ignored
    html_files = {}
    hardware_files = []
    protocol_files = []
    all_found = []  # For debug
    qpcr_files = []  # Specific for qPCR_validation_files debug
    
    for root, dirs, files in os.walk(root_dir):
        rel_root = os.path.relpath(root, root_dir).replace(os.sep, '/')  # Normalize to /
        
        # Skip ivPID directory only
        if 'ivPID' in rel_root.split('/'):
            dirs[:] = []  # Prevent further walking
            continue
        
        # No skipping _files directories - scan inside them for .html
        
        for file in files:
            if file.endswith('.html'):
                # Check for ignored basenames (case-insensitive)
                basename_stripped = os.path.splitext(file)[0].lower()
                full_rel_path = os.path.join(rel_root, file).replace(os.sep, '/')
                if basename_stripped in ignored_basenames:
                    if debug:
                        print(f"Ignored (basename match): {full_rel_path}")
                    continue  # Skip ignored files - this catches convertToHTML.html
                
                all_found.append(full_rel_path)
                basename = file.lower()  # For pattern matching
                
                # Specific debug for qPCR_validation_files
                if 'qpcr_validation_files' in rel_root:
                    qpcr_files.append(full_rel_path)
                
                if basename.startswith('instructions'):
                    hardware_files.append(full_rel_path)
                elif basename.startswith('protocol'):
                    protocol_files.append(full_rel_path)
                else:
                    # For others, group by directory for hierarchy (use / for key)
                    file_dir = rel_root if rel_root != '.' else ''
                    if file_dir not in html_files:
                        html_files[file_dir] = []
                    html_files[file_dir].append(full_rel_path)  # Fixed: Use file_dir, not dir_key (undefined)
    
    # Debug output
    if debug:
        print("All found .html files (excluding ignored like convertToHTML.html, with / paths; now including _files folders):")
        for path in sorted(all_found):
            print(f"  {path}")
        print(f"\nHardware files ({len(hardware_files)}): {sorted(hardware_files)}")
        print(f"Protocol files ({len(protocol_files)}): {sorted(protocol_files)}")
        print(f"Other files by dir: {len(html_files)} directories")
        for dir_key, flist in sorted(html_files.items()):
            if '_files' in dir_key:  # Highlight _files for verification
                print(f"  {dir_key} (from _files folder): {len(flist)} files - {sorted(flist)}")
            else:
                print(f"  {dir_key}: {len(flist)} files - {sorted(flist)[:3]}...")  # Truncate long lists
        if qpcr_files:
            print(f"\nqPCR_validation_files specific files ({len(qpcr_files)}): {sorted(qpcr_files)}")
        else:
            print("\nqPCR_validation_files: No .html files found (check if only .ipynb or excluded). Run 'find qPCR_validation_files -name \"*.html\"' to verify.")
    
    # Sort lists (ensure / paths)
    hardware_files = sorted(hardware_files)
    protocol_files = sorted(protocol_files)
    for dir_key in sorted(html_files.keys()):
        html_files[dir_key] = sorted(html_files[dir_key])
    
    total_files = len(hardware_files) + len(protocol_files) + sum(len(files_list) for files_list in html_files.values())
    
    if total_files == 0:
        print("No .html files found in the directory (excluding ignored paths).")
        return
    
    # Build HTML content with professional styling
    html_content = [
        '<!DOCTYPE html>',
        '<html lang="en">',
        '<head>',
        '    <meta charset="UTF-8">',
        '    <meta name="viewport" content="width=device-width, initial-scale=1.0">',
        '    <title>ChronoSeq Links</title>',
        '    <style>',
        '        :root { --primary-color: #2c5aa0; --secondary-color: #4a90e2; --accent-color: #7ed321; --text-color: #333; --bg-color: #f8f9fa; --card-bg: #ffffff; --shadow: 0 4px 6px rgba(0, 0, 0, 0.1); --border-radius: 8px; --transition: all 0.3s ease; }',
        '        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; line-height: 1.6; color: var(--text-color); background: linear-gradient(135deg, var(--bg-color) 0%, #e3f2fd 100%); margin: 0; padding: 20px; }',
        '        .container { max-width: 1200px; margin: 0 auto; }',
        '        h1 { text-align: center; color: var(--primary-color); font-size: 2.5rem; margin-bottom: 10px; text-shadow: 0 2px 4px rgba(0, 0, 0, 0.1); }',
        '        .subtitle { text-align: center; font-size: 1.1rem; color: #666; margin-bottom: 30px; }',
        '        .total { text-align: center; background: var(--card-bg); padding: 10px; border-radius: var(--border-radius); box-shadow: var(--shadow); display: inline-block; margin-bottom: 30px; font-weight: bold; color: var(--secondary-color); }',
        '        .section { background: var(--card-bg); margin: 20px 0; padding: 25px; border-radius: var(--border-radius); box-shadow: var(--shadow); transition: var(--transition); }',
        '        .section:hover { transform: translateY(-2px); box-shadow: 0 8px 15px rgba(0, 0, 0, 0.15); }',
        '        h2 { color: var(--primary-color); border-bottom: 2px solid var(--secondary-color); padding-bottom: 10px; margin-bottom: 20px; font-size: 1.8rem; }',
        '        h3 { color: var(--secondary-color); margin: 20px 0 15px 0; font-size: 1.3rem; border-left: 4px solid var(--accent-color); padding-left: 10px; }',
        '        ul { list-style: none; padding-left: 0; }',
        '        li { margin: 12px 0; position: relative; padding-left: 25px; }',
        '        li::before { content: "▶"; color: var(--accent-color); font-weight: bold; position: absolute; left: 0; }',
        '        a { text-decoration: none; color: var(--primary-color); font-weight: 500; padding: 8px 12px; background: rgba(74, 144, 226, 0.1); border-radius: 4px; display: inline-block; transition: var(--transition); }',
        '        a:hover { background: var(--secondary-color); color: white; transform: scale(1.05); }',
        '        a:focus { outline: 2px solid var(--accent-color); outline-offset: 2px; }',
        '        .intro { text-align: center; background: rgba(126, 211, 33, 0.1); padding: 20px; border-radius: var(--border-radius); margin-bottom: 30px; font-style: italic; color: #555; }',
        '        .analysis-desc { margin: 10px 0; color: var(--text-color); }',
        '        .analysis-desc p { margin-bottom: 15px; }',
        '        .analysis-desc ul li { list-style-type: disc; padding-left: 0; margin-left: 20px; position: relative; }',
        '        .analysis-desc ul li::before { content: none; }',
        '        @media (max-width: 768px) { body { padding: 10px; } h1 { font-size: 2rem; } .section { padding: 15px; } }',
        '    </style>',
        '</head>',
        '<body>',
        '    <div class="container">',
        f'        <h1><a href="{chronoseq_repo}">ChronoSeq</a> Links</h1>',
        f'        <p class="subtitle">Explore the key resources and documentation for the <a href="{chronoseq_repo}">ChronoSeq</a> project</p>',
        '        <p class="total">Total: ' + str(total_files) + ' pages</p>',
        f'        <div class="intro">Navigate through hardware assembly, protocols, and analysis notebooks for the <a href="{chronoseq_repo}">ChronoSeq</a> project.</div>'
    ]
    
    # Section: Main Hardware Pages
    html_content.extend([
        '        <div class="section">',
        '            <h2>Main Hardware Pages</h2>',
        '            <h3>Hardware Assembly</h3>',
        f'            <ul><li><a href="{main_hardware_links["assembly"]}">View Main Hardware Assembly Page</a></li></ul>',
        '            <h3>Hardware Operation</h3>',
        f'            <ul><li><a href="{main_hardware_links["operation"]}">View Main Hardware Operation Page</a></li></ul>',
        '        </div>'
    ])
    
    # Updated Section: Data Processing and Analysis (matching provided Markdown in HTML)
    html_content.extend([
        '        <div class="section">',
        '            <h2>Data Processing and Analysis</h2>',
        '            <div class="analysis-desc">',
        '                <ul>',
        '                    <li>Check out the <a href="https://github.com/kanishkasthana/ChronoSeq-Tools">ChronoSeq-Tools repo</a> to help process your raw data.</li>',
        '                    <li>Check out the <a href="https://github.com/kanishkasthana/ChronoSeq-QC">ChronoSeq-QC repo</a> to see how the QC plots were generated.</li>',
        '                    <li>Check out the <a href="https://github.com/anjambor/ChronoSeq-Analysis">ChronoSeq-Analysis repo</a> to see how the ChronoSeq data was analysed.</li>',
        '                </ul>',
        '            </div>',
        '        </div>'
    ])
    
    # Section: ChronoSeq Hardware Assembly (flat)
    if hardware_files:
        html_content.extend([
            '        <div class="section">',
            f'            <h2><a href="{chronoseq_repo}">ChronoSeq</a> Hardware Assembly</h2>',
            '            <ul>'
        ])
        for rel_path in hardware_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20 for display
            html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        html_content.extend(['            </ul>', '        </div>'])
    
    # Section: ChronoSeq Protocols (flat)
    if protocol_files:
        html_content.extend([
            '        <div class="section">',
            f'            <h2><a href="{chronoseq_repo}">ChronoSeq</a> Protocols</h2>',
            '            <ul>'
        ])
        for rel_path in protocol_files:
            link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20
            html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        html_content.extend(['            </ul>', '        </div>'])
    
    # Section: Other Notebooks (hierarchical by dir, with /)
    if html_files:
        html_content.extend([
            '        <div class="section">',
            '            <h2>Other Notebooks</h2>'
        ])
        sorted_dirs = sorted(html_files.keys())
        current_dir = ""
        for dir_key in sorted_dirs:
            if dir_key != current_dir:
                if current_dir:
                    html_content.append('            </ul>')
                if dir_key:
                    dir_display = dir_key.replace('/', ' > ')
                    html_content.extend([
                        f'            <h3>{dir_display}</h3>',
                        '            <ul>'
                    ])
                else:
                    html_content.extend([
                        '            <h3>Root Directory</h3>',
                        '            <ul>'
                    ])
                current_dir = dir_key
            
            for rel_path in html_files[dir_key]:
                link_text = os.path.basename(rel_path).replace('.html', '').replace('%20', ' ')  # Decode %20 for readability
                html_content.append(f'                <li><a href="{rel_path}">{link_text}</a></li>')
        
        html_content.append('            </ul>')  # Close final ul
        html_content.append('        </div>')
    
    html_content.extend([
        '    </div>',
        '</body>',
        '</html>'
    ])
    
    # Write to sitemap.html in root
    output_path = os.path.join(root_dir, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_content))
    
    print(f"Generated professional sitemap.html with {total_files} HTML files at: {output_path}")
    print("Updated intro text and linked all 'ChronoSeq' mentions to https://github.com/kanishkasthana/ChronoSeq.")
    print("convertToHTML.html excluded. Styling modern and responsive. _files scanned; qPCR_validation_files under 'Other Notebooks'.")

# Example usage: Replace '.' with your directory path
directory = '.'  # Current directory; change as needed
generate_sitemap_html(directory, debug=False)  # Set debug=True if needed for verification


Generated professional sitemap.html with 40 HTML files at: C:\Users\ChronoSeq\ChronoSeq\sitemap.html
Updated intro text and linked all 'ChronoSeq' mentions to https://github.com/kanishkasthana/ChronoSeq.
convertToHTML.html excluded. Styling modern and responsive. _files scanned; qPCR_validation_files under 'Other Notebooks'.
